# Colab Setup — COS40007 Structural Defect Detection

Run this notebook **once at the start of every Colab session** before training.

**Order:**
1. Mount Google Drive
2. Navigate into the repo
3. Pull latest changes
4. Install dependencies
5. Verify GPU
6. Train the model you need (one per session)
7. Push weights back to GitHub

## Step 1 — Mount Google Drive

`force_remount=True` reconnects a dropped Drive — safe to run every session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
repo = '/content/drive/MyDrive/COS40007/ai_for_engineering'
print('Drive mounted:', os.path.exists('/content/drive/MyDrive'))
print('Repo exists: ', os.path.exists(repo))

## Step 2 — Clone repo (FIRST TIME ONLY)

If the cell above shows **Repo exists: True**, skip this cell entirely.

Only uncomment and run if you have never cloned the repo to this Drive account.

In [ ]:
# FIRST TIME ONLY — uncomment the line below and run it once
# !git clone https://github.com/tylsee/ai_for_engineering.git /content/drive/MyDrive/COS40007/ai_for_engineering

## Step 3 — Navigate into the repo and pull latest

In [ ]:
%cd /content/drive/MyDrive/COS40007/ai_for_engineering
!pwd
!git pull origin main

## Step 4 — Install dependencies

Colab already has PyTorch with CUDA — only install the rest.

In [ ]:
!pip install -q ultralytics albumentations opencv-python Pillow pandas matplotlib \
    seaborn scikit-learn streamlit pyyaml torchmetrics tqdm \
    pycocotools faster-coco-eval pypdf

## Step 5 — Verify GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 6 — Train

**Run only ONE of the cells below per session** — training one model at a time avoids losing progress if Colab disconnects.

Weights are saved to `runs/` inside your Drive folder and survive disconnections.

| Model | Est. time (T4) | Output |
|-------|----------------|--------|
| YOLOv11n | ~60–90 min | `runs/yolo11n/v1/weights/best.pt` |
| YOLOv8n  | ~60–90 min | `runs/yolov8n/v1/weights/best.pt` |
| SSDLite  | ~30–45 min | `runs/ssdlite/best_v1.pth` |

In [ ]:
# ── Train YOLOv11n ────────────────────────────────────────────────────────────
# Session 1 — run this cell only
!python scripts/train_all.py --model yolo11n

In [ ]:
# ── Train YOLOv8n ─────────────────────────────────────────────────────────────
# Session 2 — run this cell only
!python scripts/train_all.py --model yolov8n

In [ ]:
# ── Train SSDLite-MobileNetV3 ─────────────────────────────────────────────────
# Session 3 — run this cell only
!python scripts/train_all.py --model ssdlite

## Step 7 — Push trained weights to GitHub

Run after training completes. Replace `YOUR_TOKEN` with your GitHub personal access token (classic, `repo` scope).

In [ ]:
%cd /content/drive/MyDrive/COS40007/ai_for_engineering
!git add runs/
!git commit -m "Add trained weights from Colab session"
!git push https://YOUR_TOKEN@github.com/tylsee/ai_for_engineering.git main

---
## Troubleshooting — Drive disconnected (OSError 107)

If you see `Transport endpoint is not connected`, do this:

1. Re-run **Step 1** (the mount cell with `force_remount=True`)
2. Re-run **Step 3** (`%cd` + `git pull`)
3. Re-run your training cell

Do **not** restart the runtime — that kills your training progress.

---
## Quick reference

```bash
# Smoke test before training (~10 sec)
!python scripts/smoke_test.py

# Check what runs are already saved
!ls runs/yolo11n/ runs/yolov8n/ runs/ssdlite/ 2>/dev/null

# View run log
import pandas as pd
pd.read_csv('runs/run_log.csv')
```